# Run11 Relaxed-Risk Training
Thin orchestration notebook for the canonical Run11 relaxed-risk alpha-generation training path. Durable logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/env modules.


## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [1]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

TRAIN_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
TRAIN_REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")
TRAIN_BRANCH = "feature/run9-alpha-overhaul-20260311"
INSTALL_REQUIREMENTS = True

def run(cmd):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

run(["git", "ls-remote", "--exit-code", "--heads", TRAIN_REPO_URL, TRAIN_BRANCH])

if not (TRAIN_REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", TRAIN_BRANCH, TRAIN_REPO_URL, str(TRAIN_REPO_DIR)])
else:
    run(["git", "-C", str(TRAIN_REPO_DIR), "fetch", "origin"])
    run(["git", "-C", str(TRAIN_REPO_DIR), "checkout", TRAIN_BRANCH])
    run(["git", "-C", str(TRAIN_REPO_DIR), "reset", "--hard", f"origin/{TRAIN_BRANCH}"])

purge_paths = [
    TRAIN_REPO_DIR / "tcn_fusion_results",
    TRAIN_REPO_DIR / "tcn_results",
    TRAIN_REPO_DIR / "tcn_att_results",
    TRAIN_REPO_DIR / "output_log",
    TRAIN_REPO_DIR / "output_logs",
    TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
    TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
    TRAIN_REPO_DIR / "data" / "daily_ohlcv_assets.csv",
    TRAIN_REPO_DIR / "data" / "processed_daily_macro_features.csv",
]

for path in purge_paths:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.exists():
        path.unlink()

for cache_dir in TRAIN_REPO_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)

for ckpt_dir in TRAIN_REPO_DIR.rglob(".ipynb_checkpoints"):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

if INSTALL_REQUIREMENTS:
    requirements_file = TRAIN_REPO_DIR / "requirements.txt"
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])

print("[OK] Repo synced:", TRAIN_REPO_DIR)
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "HEAD"])
print("[OK] Requirements installed:", INSTALL_REQUIREMENTS)

+ git ls-remote --exit-code --heads https://github.com/Dave-DKings/tcn_tape_vectorized_version.git feature/run9-alpha-overhaul-20260311
+ git -C /content/tcn_tape_vectorized_version_clean fetch origin
+ git -C /content/tcn_tape_vectorized_version_clean checkout feature/run9-alpha-overhaul-20260311
+ git -C /content/tcn_tape_vectorized_version_clean reset --hard origin/feature/run9-alpha-overhaul-20260311
+ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
+ /usr/bin/python3 -m pip install -r /content/tcn_tape_vectorized_version_clean/requirements.txt
[OK] Repo synced: /content/tcn_tape_vectorized_version_clean
+ git -C /content/tcn_tape_vectorized_version_clean rev-parse --abbrev-ref HEAD
+ git -C /content/tcn_tape_vectorized_version_clean rev-parse HEAD
[OK] Requirements installed: True


In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
if not gpus:
    raise RuntimeError('No GPU visible to TensorFlow. In Colab: Runtime -> Change runtime type -> GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())


TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run11_relaxed_config, assert_run11_relaxed_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import prepare_phase1_dataset, run_experiment6_tape

RUN_ID = 'run11'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None


## 3) Build Canonical Run11 Config and Dataset
Create the source-backed Run11 relaxed-risk config, assert no drift, and prepare the dataset once.


In [4]:
train_config = build_run11_relaxed_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
assert_run11_relaxed_config(train_config)

tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']

print('[Run11] Canonical relaxed-risk config ready')
print('  analysis_start_date =', train_config['ANALYSIS_START_DATE'])
print('  split_date =', train_config['TRAIN_TEST_SPLIT_DATE'])
print('  architecture =', ap['actor_critic_type'])
print('  regime_conditioning =', ap['regime_conditioning_enabled'])
print('  distributional_critic =', ap['distributional_critic_enabled'])
print('  cvar_advantage_weight =', ppo['cvar_advantage_weight'])
print('  lagrangian =', {
    'threshold': ppo['lagrangian_cvar_threshold'],
    'lr': ppo['lagrangian_cvar_lr'],
    'lambda_max': ppo['lagrangian_cvar_lambda_max'],
    'penalty_scale': ppo['lagrangian_cvar_penalty_scale'],
})
print('  drawdown =', {
    'target': env['drawdown_constraint']['target'],
    'tolerance': env['drawdown_constraint']['tolerance'],
    'penalty_coef': env['drawdown_constraint']['penalty_coef'],
    'lambda_carry_decay': env['drawdown_constraint']['lambda_carry_decay'],
    'terminal_gate_a_max_drawdown': env['tape_terminal_gate_a_max_drawdown'],
})
print('  turnover =', {
    'target': env['target_turnover'],
    'base_scalar': env['turnover_penalty_scalar'],
    'curriculum': tp['turnover_penalty_curriculum'],
})
print('  execution_beta =', {
    'train_curriculum': tp['action_execution_beta_curriculum'],
    'eval_beta': tp['evaluation_action_execution_beta'],
})
print('  dispersion =', {
    'hhi_coef': ppo['alpha_diversity_coef'],
    'dispersion_coef': ppo['alpha_dispersion_coef'],
    'dispersion_target_std': ppo['alpha_dispersion_target_std'],
})
print('  checkpointing =', {
    'periodic_every_steps': tp['periodic_checkpoint_every_steps'],
    'high_watermark_sharpe_threshold': tp['high_watermark_sharpe_threshold'],
    'high_watermark_max_drawdown_abs_threshold': tp['high_watermark_max_drawdown_abs_threshold'],
    'step_sharpe_threshold': tp['step_sharpe_checkpoint_threshold'],
})
print('  deterministic_validation =', tp['deterministic_validation_checkpointing_enabled'])
print('  training_early_stop =', tp.get('training_early_stop_enabled', False))

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Actuarial_')]
if actuarial_cols:
    raise RuntimeError(f'Actuarial columns should be absent for Run11: {actuarial_cols}')

alpha_ret_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('AlphaRet_')]
expected_alpha_cols = {'AlphaRet_1d', 'AlphaRet_5d', 'AlphaRet_20d', 'AlphaRet_5d_Z', 'AlphaRet_20d_Z'}
missing_alpha_cols = sorted(list(expected_alpha_cols - set(alpha_ret_cols)))
if missing_alpha_cols:
    raise RuntimeError(f'Missing expected alpha-return columns: {missing_alpha_cols}')

fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Fundamental_')]
if fundamental_cols:
    raise RuntimeError(f'Fundamental columns should be absent: {fundamental_cols}')

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Actuarial feature check passed: none present (disabled)')
print('[OK] Alpha-return feature check passed:', sorted(alpha_ret_cols)[:10])
print('[OK] Fundamental feature check passed: none present')


[Run10] Canonical config ready
  analysis_start_date = 2016-01-01
  split_date = 2021-12-31
  architecture = TCN_FUSION
  regime_conditioning = True
  distributional_critic = True
  cvar_advantage_weight = 0.1
  lagrangian = {'threshold': -0.025, 'lr': 0.004, 'lambda_max': 5.0, 'penalty_scale': 3.0}
  drawdown = {'target': 0.18, 'tolerance': -0.015, 'penalty_coef': 1.5, 'lambda_carry_decay': 0.75}
  dispersion = {'hhi_coef': 0.01, 'dispersion_coef': 0.05, 'dispersion_target_std': 0.07}
  deterministic_validation = False
  training_early_stop = True
📊 Loading raw market data...
   [OK] Raw data shape: (55043, 7)
   [OK] Date range: 2003-09-02 00:00:00 => 2025-08-29 00:00:00

[TOOL] Computing multi-horizon log returns: [1, 5, 10, 21]
   [OK] Shape after returns: (54833, 11)

📈 Calculating 21-day rolling statistics

🧮 Computing technical indicators

🕯️ Adding candlestick geometry features (if enabled)

📊 Computing dynamic covariance features

🎯 Adding regime awareness features
   [OK] Mas

## 4) Run Training
Launch the canonical Run11 relaxed-risk training path.


In [5]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')


[START] Starting training
Architecture: TCN_FUSION
max_total_timesteps: 500000
num_parallel_envs: 4

EXPERIMENT 6: TCN_FUSION Enhanced + TAPE Three-Component
Architecture: TCN + Fusion
Results root: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results
Working dir: /content/tcn_tape_vectorized_version_clean
Covariance Features: Yes
🎯 REWARD SYSTEM: TAPE (Three-Component v3)
   Profile: BalancedGrowth
   Daily: Base + DSR/PBRS + Turnover_Proximity
   Terminal: mode=signed | baseline=0.20 | scalar=10.0 (clipped ±10.0)
   Gate A: enabled (Sharpe <= 0.00 or MDD >= 25.0% -> force non-positive terminal bonus)
   Neutral Band: enabled (±0.020 around baseline)
   [CYCLE] Profile Manager: disabled (static profile only)
[RAND] Experiment Seed: 6042 (Base: 42, Offset: 6000)
[OK] Features: Enhanced (includes 2 covariance eigenvalues)
   Eigenvalues: ['Covariance_Eigenvalue_0', 'Covariance_Eigenvalue_1']
   Train shape: (15110, 67)
   Test shape: (9180, 67)
   ℹ️ Actuarial features disabled

## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [6]:
TRAIN_RESULTS_ROOT = TRAIN_REPO_DIR / 'tcn_fusion_results'
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


Episodes file: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260313_055213_episodes.csv
Rows: 119


,update,timestep,episode,elapsed_time,episode_return_pct,episode_sharpe,episode_sortino,episode_max_dd,episode_volatility,episode_win_rate,...,explained_variance,actor_grad_norm,critic_grad_norm,alpha_min,alpha_max,alpha_mean,alpha_cap_hit_frac,ratio_mean,ratio_std,drawdown_lambda_peak
99,100,100800,128,5347.103928,59.507136,0.684383,0.781472,27.818635,0.188351,57.063404,...,0.939536,1.423367,0.323341,0.776118,2.116785,1.311142,0.0,0.929703,2.451816,0.027147
100,101,101808,130,5400.609791,69.401530,0.732294,0.894310,24.383358,0.183598,55.325750,...,0.927997,1.254053,0.349891,0.832329,2.162570,1.238821,0.0,1.057877,3.591024,0.036090
101,102,102816,132,5453.971004,73.296632,0.737701,0.836185,27.227249,0.182370,56.405164,...,0.913525,1.326332,0.414890,0.925264,1.921234,1.282567,0.0,1.074485,3.606202,0.080436
102,103,103824,132,5507.913365,57.445277,1.534500,2.028997,8.832721,0.106000,56.877898,...,0.918279,1.668184,0.497788,0.918076,1.790337,1.192616,0.0,1.074857,3.461167,0.080436
103,104,104832,132,5561.761507,50.759485,0.839062,1.073296,18.784261,0.122512,56.284761,...,0.932671,1.361065,0.879080,0.887774,1.818504,1.116009,0.0,1.053783,3.106730,0.080436
104,105,105840,134,5616.409622,43.378985,0.509729,0.579168,27.261356,0.164765,55.412115,...,0.932354,1.104703,0.231742,0.733132,1.730751,0.967926,0.0,1.015504,2.736145,0.059222
105,106,106848,136,5671.609739,63.182356,0.656313,0.748759,27.422990,0.181868,56.504469,...,0.927724,1.168565,0.164439,0.645466,2.038047,0.912607,0.0,0.925464,1.959766,0.091534
106,107,107856,136,5725.980523,18.562089,0.331409,0.403097,22.585116,0.201626,54.404946,...,0.940771,1.357265,0.425095,0.589667,2.097779,0.865554,0.0,1.055430,2.688126,0.091534
107,108,108864,136,5779.475253,47.233451,0.575412,0.696836,22.585116,0.183237,56.173526,...,0.945698,1.408439,0.407277,0.731949,2.000364,0.969604,0.0,0.951545,2.013271,0.091534
108,109,109872,138,5833.502748,67.745874,0.701795,0.823028,25.576983,0.179177,56.504469,...,0.939203,1.502077,0.468812,0.847958,2.212161,1.097595,0.0,0.996500,2.470634,0.044417


Step diagnostics file: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260313_055213_step_diagnostics.csv
Rows: 119952


,update,timestep,episode,episode_step,date,elapsed_time,reward_total,portfolio_return_pct_points,portfolio_value,prev_portfolio_value,...,turnover_penalty_contrib,transaction_cost_dollars,tx_cost_contrib_reward_pts,action_realization_l1,action_realization_penalty,cash_weight_raw,cash_weight_projected,cash_weight_final,cash_weight_forced_gap,drawdown_penalty
119932,119,119933,148,643,2018-09-14 00:00:00,6354.920041,-0.077329,0.041743,168233.814848,168163.618907,...,-0.066536,66.030993,-0.039266,0.090183,0.045091,0.022944,0.050000,0.076315,0.027056,0.0
119933,119,119934,148,496,2018-08-09 00:00:00,6354.922971,0.545454,-0.327850,138375.933203,138831.091094,...,0.000000,40.113488,-0.028894,0.000000,0.000000,0.072594,0.072594,0.077600,0.000000,0.0
119934,119,119935,148,565,2019-03-04 00:00:00,6354.925943,0.095913,0.237845,128680.234124,128374.901208,...,0.000000,31.995178,-0.024923,0.059009,0.029505,0.020495,0.050000,0.096189,0.029505,0.0
119935,119,119936,148,462,2018-01-29 00:00:00,6354.928734,-0.374158,-0.342455,155946.247473,156482.128441,...,0.000000,29.303703,-0.018727,0.028417,0.014209,0.035791,0.050000,0.063358,0.014209,0.0
119936,119,119937,148,644,2018-09-17 00:00:00,6355.080742,0.272965,-0.123863,168025.436092,168233.814848,...,0.000000,51.804627,-0.030793,0.219402,0.109701,0.013885,0.050000,0.064473,0.036115,0.0
119937,119,119938,148,497,2018-08-10 00:00:00,6355.083873,-0.691687,-0.474743,137719.002724,138375.933203,...,0.000000,39.103655,-0.028259,0.083374,0.041687,0.084167,0.084167,0.080555,0.000000,0.0
119938,119,119939,148,566,2019-03-05 00:00:00,6355.086916,0.927953,-0.193915,128430.703489,128680.234124,...,0.000000,32.672980,-0.025391,0.020030,0.010015,0.039985,0.050000,0.075404,0.010015,0.0
119939,119,119940,148,463,2018-01-30 00:00:00,6355.089729,-0.711819,-0.487958,155185.294805,155946.247473,...,0.000000,39.235384,-0.025160,0.000000,0.000000,0.064383,0.064383,0.063819,0.000000,0.0
119940,119,119941,148,645,2018-09-18 00:00:00,6355.240436,0.120822,0.043013,168097.709012,168025.436092,...,0.000000,25.912232,-0.015422,0.042940,0.021470,0.028530,0.050000,0.057960,0.021470,0.0
119941,119,119942,148,498,2018-08-13 00:00:00,6355.243268,-0.496929,-0.422479,137137.168705,137719.002724,...,0.000000,33.253972,-0.024146,0.059217,0.029608,0.020392,0.050000,0.066805,0.029608,0.0


## 6) Export Artifacts (Optional)
Zip the latest results and optionally copy them to Google Drive.


In [8]:
import subprocess

EXPORT_RESULTS_ZIP = True
COPY_TO_DRIVE = True
EXPORT_PATH = Path(f'/content/tcn_tape_vectorized_{RUN_ID}.zip')

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_REPO_DIR / 'tcn_fusion_results',
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data_exports',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            ['bash', '-lc', 'cd "{}" && zip -qr "{}" {}'.format(TRAIN_REPO_DIR, EXPORT_PATH, ' '.join(f'"{item}"' for item in relative_items))],
            check=True,
        )
        print('[OK] Created:', EXPORT_PATH)

        if COPY_TO_DRIVE:
            from google.colab import drive
            drive.mount('/content/drive')
            subprocess.run(['cp', str(EXPORT_PATH), '/content/drive/MyDrive/'], check=True)
            print('[OK] Copied to Drive:', f'/content/drive/MyDrive/{EXPORT_PATH.name}')
    else:
        print('[WARN] Nothing to export.')
else:
    print('[SKIP] EXPORT_RESULTS_ZIP=False')


Mounted at /content/drive
[OK] Copied to Drive: /content/drive/MyDrive/tcn_tape_vectorized_run10.zip


In [9]:
from pathlib import Path
import csv, re

root = Path("/content/tcn_tape_vectorized_version_clean/tcn_fusion_results")
log_dir = root / "logs"
ckpt_dir = root / "high_watermark_checkpoints"

# latest episodes csv
ep_csv = sorted(log_dir.glob("*_episodes.csv"), key=lambda p: p.stat().st_mtime)[-1]
print("episodes_csv:", ep_csv)

rows = list(csv.DictReader(ep_csv.open()))
print("rows:", len(rows))
if rows:
    last = rows[-1]
    for k in ["update", "step", "episode", "episode_sharpe", "episode_return_pct", "episode_turnover_pct"]:
        if k in last:
            print(k, "=", last[k])

# best checkpoint by Sharpe tag in filename
pat = re.compile(r"shp([pm]?\d+p\d+)")
best = None
for f in ckpt_dir.glob("*_actor.weights.h5"):
    m = pat.search(f.name)
    if not m:
        continue
    t = m.group(1)
    sign = -1 if t.startswith("m") else 1
    t = t[1:] if t[0] in "pm" else t
    sh = sign * float(t.replace("p", "."))
    if best is None or sh > best[0]:
        best = (sh, f)

print("best_checkpoint:", best[1] if best else None)
print("best_sharpe_tag:", best[0] if best else None)

episodes_csv: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260313_055213_episodes.csv
rows: 119
update = 119
episode = 148
episode_sharpe = 1.8630986757129058
episode_return_pct = 68.64746699151563
episode_turnover_pct = 26.506832242012024
best_checkpoint: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00008_shp1p444_actor.weights.h5
best_sharpe_tag: 1.444


In [ ]:
# Rank all saved checkpoints by deterministic eval Sharpe, then run stochastic on top-K

from pathlib import Path
import copy
import pandas as pd
import numpy as np

# ---- knobs ----
RESULTS_ROOT = Path("/content/tcn_tape_vectorized_version_clean/tcn_fusion_results")
HORIZON_DAYS = 756          # change to 252/504/756/1008
DET_SEED = 6042
TOP_K = 3
STOCH_RUNS_TOPK = 20        # set 0 to skip stochastic pass

# ---- resolve objects from notebook globals ----
exp_obj = globals().get("train_experiment6") or globals().get("experiment6")
phase1_obj = globals().get("eval_phase1_data") or globals().get("train_phase1_data")
base_cfg = globals().get("eval_config") or globals().get("train_config")

if exp_obj is None or phase1_obj is None or base_cfg is None:
    raise RuntimeError("Missing one of: experiment6/train_experiment6, eval_phase1_data/train_phase1_data, eval_config/train_config")

# if helper not in scope yet
if "evaluate_experiment6_checkpoint" not in globals():
    from src.notebook_helpers.tcn_phase1 import evaluate_experiment6_checkpoint

cfg = copy.deepcopy(base_cfg)

# ---- discover checkpoint prefixes ----
prefixes = []
for actor_path in (RESULTS_ROOT / "high_watermark_checkpoints").glob("*_actor.weights.h5"):
    prefix = str(actor_path).replace("_actor.weights.h5", "")
    critic_path = Path(prefix + "_critic.weights.h5")
    if critic_path.exists():
        prefixes.append(prefix)

prefixes = sorted(set(prefixes))
print(f"Found {len(prefixes)} candidate checkpoints")

# ---- deterministic ranking ----
det_rows = []
for i, prefix in enumerate(prefixes, 1):
    try:
        er = evaluate_experiment6_checkpoint(
            experiment6=exp_obj,
            phase1_data=phase1_obj,
            config=cfg,
            random_seed=DET_SEED,
            checkpoint_path_override=prefix,
            deterministic_eval_mode="mean",
            stochastic_eval_mode="sample",
            num_eval_runs=0,
            stochastic_episode_length_limit=HORIZON_DAYS,
            save_eval_logs=False,
            save_eval_artifacts=False,
        )
        dm = er.deterministic_metrics or {}
        det_rows.append({
            "prefix": prefix,
            "det_sharpe": float(dm.get("sharpe_ratio", np.nan)),
            "det_return_pct": float(dm.get("total_return_pct", np.nan)),
            "det_mdd_pct": float(dm.get("max_drawdown_pct", np.nan)),
            "det_turnover_pct": float(dm.get("turnover", np.nan)) * 100.0 if dm.get("turnover", None) is not None else np.nan,
        })
    except Exception as e:
        det_rows.append({"prefix": prefix, "error": f"{type(e).__name__}: {e}"})

det_df = pd.DataFrame(det_rows)
ok_df = det_df[det_df["det_sharpe"].notna()].copy()
ranked_df = ok_df.sort_values(["det_sharpe", "det_return_pct", "det_mdd_pct"], ascending=[False, False, True]).reset_index(drop=True)

print("\nTop deterministic checkpoints")
display(ranked_df.head(TOP_K))

# ---- stochastic pass on top-K ----
if STOCH_RUNS_TOPK > 0 and not ranked_df.empty:
    st_rows = []
    for _, r in ranked_df.head(TOP_K).iterrows():
        prefix = r["prefix"]
        er = evaluate_experiment6_checkpoint(
            experiment6=exp_obj,
            phase1_data=phase1_obj,
            config=cfg,
            random_seed=DET_SEED + 100000,
            checkpoint_path_override=prefix,
            deterministic_eval_mode="mean",
            stochastic_eval_mode="sample",
            num_eval_runs=STOCH_RUNS_TOPK,
            stochastic_episode_length_limit=HORIZON_DAYS,
            save_eval_logs=False,
            save_eval_artifacts=False,
        )
        sto = er.stochastic_results if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        st_rows.append({
            "prefix": prefix,
            "det_sharpe": r["det_sharpe"],
            "sto_sharpe_mean": float(sto["sharpe_ratio"].mean()) if "sharpe_ratio" in sto else np.nan,
            "sto_sharpe_std": float(sto["sharpe_ratio"].std()) if "sharpe_ratio" in sto else np.nan,
            "sto_return_mean_pct": float(sto["total_return"].mean() * 100.0) if "total_return" in sto else np.nan,
            "sto_mdd_mean_pct": float(sto["max_drawdown"].mean() * 100.0) if "max_drawdown" in sto else np.nan,
        })

    st_df = pd.DataFrame(st_rows).sort_values(["sto_sharpe_mean", "det_sharpe"], ascending=[False, False]).reset_index(drop=True)
    print("\nTop-K stochastic rerank")
    display(st_df)

Found 60 candidate checkpoints

LOADING CUSTOM CHECKPOINT: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00002_shp1p153
[OK] Found actor weights: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00002_shp1p153_actor.weights.h5
[OK] Found critic weights: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00002_shp1p153_critic.weights.h5
🏗️ Recreating evaluation environments...
   🧭 Checkpoint architecture: TCN_FUSION (attention=True, fusion=True, source=path)
   🧱 Eval TCN stack: filters=[64, 96, 128, 128, 128] | kernel=5 | dilations=[1, 2, 4, 8, 16] | dropout=0.15
   🧩 Eval fusion core: embed=128 | heads=4 | dropout=0.1
   🔀 Eval mixer (A4): enabled=False | layers=1 | expansion=2.0 | dropout=0.1
   🎯 Eval alpha head (A3): dims=[128, 64] | dropout=0.05
   🧭 Eval fusion v2 cross-attn: asset_identity=True | context_cross_attn

,prefix,det_sharpe,det_return_pct,det_mdd_pct,det_turnover_pct
0,/content/tcn_tape_vectorized_version_clean/tcn...,1.044581,NaN,NaN,0.726482
1,/content/tcn_tape_vectorized_version_clean/tcn...,1.044581,NaN,NaN,0.726482
2,/content/tcn_tape_vectorized_version_clean/tcn...,1.008787,NaN,NaN,0.384217



LOADING CUSTOM CHECKPOINT: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00002_shp1p153
[OK] Found actor weights: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00002_shp1p153_actor.weights.h5
[OK] Found critic weights: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00002_shp1p153_critic.weights.h5
🏗️ Recreating evaluation environments...
   🧭 Checkpoint architecture: TCN_FUSION (attention=True, fusion=True, source=path)
   🧱 Eval TCN stack: filters=[64, 96, 128, 128, 128] | kernel=5 | dilations=[1, 2, 4, 8, 16] | dropout=0.15
   🧩 Eval fusion core: embed=128 | heads=4 | dropout=0.1
   🔀 Eval mixer (A4): enabled=False | layers=1 | expansion=2.0 | dropout=0.1
   🎯 Eval alpha head (A3): dims=[128, 64] | dropout=0.05
   🧭 Eval fusion v2 cross-attn: asset_identity=True | context_cross_attn=False | ctx_heads=4 | ctx_drop

,prefix,det_sharpe,sto_sharpe_mean,sto_sharpe_std,sto_return_mean_pct,sto_mdd_mean_pct
0,/content/tcn_tape_vectorized_version_clean/tcn...,1.044581,0.621843,0.189013,39.257935,20.826741
1,/content/tcn_tape_vectorized_version_clean/tcn...,1.044581,0.611593,0.214199,38.515229,21.509074
2,/content/tcn_tape_vectorized_version_clean/tcn...,1.008787,-0.062290,0.240692,-0.121749,23.835598
